## Introduction

In this project, we will be analyzing some data from a sample of anonymous patients who were evaluated for heart disease at the `Cleveland Clinic Foundation`. This processed data has been downloaded from the `UCI Machine Learning Repository`.

In this project our primary areas of focus will be as followed:

- Analyzing `thalach` to understand some of the more accurate predictors of heart disease
- Investigating the relationship between maximum heart rate and the kind of chest pain that a patient experiences
- Investigating the relationship between the kind of chest pain that a patient experiences and if they have heart disease or not

### Data Features

The following list will introduce the features (columns) of our dataset and give a brief explanation of each feature:

- `age`: The patient's age in years
- `sex`: The patient's sex assigned at birth (`1` for male and `0` for female)
- `trestbps`: The patient's resting blood pressure in mm Hg
- `chol`: The patient's serum cholseterol in mg/dl
- `cp`: The kind of chest pain that the patient is experiencing (`1`: typical angina, `2`: atypical angina, `3`: non-anginal pain, `4`: asymptomatic)
- `exang`: Whether the patient experiences exercise-induced angina (`1`: yes, `0`: no)
- `fbs`: Whether the patient's fasting blood sugar is > 120 mg/dl (`1`: yes, `0`: no)
- `thalach`: The patient's maximum heart rate achieved in exercise test
- `heart_disease`: Whether the patient is found to have heart disease (`1`: yes, `0`: no)

In [ ]:
#Installing Necessary Packages
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install seaborn
%pip install scipy
%pip install statsmodels

In [2]:
#Importing the Installed Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
from scipy.stats import f_oneway
from scipy.stats import chi2_contingency
from statsmodels.stats.multicomp import pairwise_tukeyhsd

%matplotlib inline


## Formatting the Data

The first thing that we need to do is load our data set into a pandas DataFrame.

Here, our intial DataFrame will be named, `heart`.

In [3]:
# Create a path to the data file because it is not a .csv extention
file_path = 'processed.cleveland.data'

# Read the data into a pandas DataFrame
heart = pd.read_csv(file_path, delimiter=',', header=None)

#Display the first few rows of our new DataFrame
print(heart.head())

     0    1    2      3      4    5    6      7    8    9    10   11   12  13
0  63.0  1.0  1.0  145.0  233.0  1.0  2.0  150.0  0.0  2.3  3.0  0.0  6.0   0
1  67.0  1.0  4.0  160.0  286.0  0.0  2.0  108.0  1.0  1.5  2.0  3.0  3.0   2
2  67.0  1.0  4.0  120.0  229.0  0.0  2.0  129.0  1.0  2.6  2.0  2.0  7.0   1
3  37.0  1.0  3.0  130.0  250.0  0.0  0.0  187.0  0.0  3.5  3.0  0.0  3.0   0
4  41.0  0.0  2.0  130.0  204.0  0.0  2.0  172.0  0.0  1.4  1.0  0.0  3.0   0


From this initial DataFrame, we can see that our column headers are empty as we had specified not to have any when creating our DataFrame. Let's fix that.

In [4]:
# Defining column headers
column_names = ['age', 'sex', 'cp type', 'trestbps', 'chol', 'fbs', 'u1', 'thalach', 
                'exang', 'u2', 'u3', 'u4', 'u5', 'heart_disease']

#Assigning the headers to our DataFrame
heart.columns = column_names

#Displaying our updated DataFrame
print(heart.head())



    age  sex  cp type  trestbps   chol  fbs   u1  thalach  exang   u2   u3  \
0  63.0  1.0      1.0     145.0  233.0  1.0  2.0    150.0    0.0  2.3  3.0   
1  67.0  1.0      4.0     160.0  286.0  0.0  2.0    108.0    1.0  1.5  2.0   
2  67.0  1.0      4.0     120.0  229.0  0.0  2.0    129.0    1.0  2.6  2.0   
3  37.0  1.0      3.0     130.0  250.0  0.0  0.0    187.0    0.0  3.5  3.0   
4  41.0  0.0      2.0     130.0  204.0  0.0  2.0    172.0    0.0  1.4  1.0   

    u4   u5  heart_disease  
0  0.0  6.0              0  
1  3.0  3.0              2  
2  2.0  7.0              1  
3  0.0  3.0              0  
4  0.0  3.0              0  


Now, you may notice that there are columns labeled `u1`, `u2`, `u3`, `u4`, and `u5`.

These are simply columns to which the values stored will not be needed for the analysis conducted in this project.

So let's just go ahead and remove those columns.

In [5]:
# Defining the columns that we want to be removed
columns_to_remove = ['u1', 'u2', 'u3', 'u4', 'u5']

# Removing the columns from our DataFrame
heart = heart.drop(columns = columns_to_remove)

#Displaying our updated DataFrame
print(heart.head())

    age  sex  cp type  trestbps   chol  fbs  thalach  exang  heart_disease
0  63.0  1.0      1.0     145.0  233.0  1.0    150.0    0.0              0
1  67.0  1.0      4.0     160.0  286.0  0.0    108.0    1.0              2
2  67.0  1.0      4.0     120.0  229.0  0.0    129.0    1.0              1
3  37.0  1.0      3.0     130.0  250.0  0.0    187.0    0.0              0
4  41.0  0.0      2.0     130.0  204.0  0.0    172.0    0.0              0


Last but not least within our data preparation, we will take care of the heart disease column. Here, we have values ranging from 0-4 where a value of `0` means that a patient does not have detectable heart disease, whereas any number greater than zero up to four (`1`, `2`, `3`, `4`) not only means that a patient has detectable hear disease, but also represents the severity of the heart disease with a value of `4` being the most severe.

In our analysis here, we will only be analyzing whethere a patient has heart disease or not, and we will not be taking into account how severe that heart disease is.

Therefore, we need the heart disease column to be binary, `0` for no detectable heart disease or `1` for detectable heart disease. Let's update our column to contain that format.

In [6]:
# Update the column using a lambda function
heart['heart_disease'] = heart['heart_disease'].apply(lambda x: 1 if x > 0 else 0)

# Displaying our updated DataFrame
print(heart.head())

    age  sex  cp type  trestbps   chol  fbs  thalach  exang  heart_disease
0  63.0  1.0      1.0     145.0  233.0  1.0    150.0    0.0              0
1  67.0  1.0      4.0     160.0  286.0  0.0    108.0    1.0              1
2  67.0  1.0      4.0     120.0  229.0  0.0    129.0    1.0              1
3  37.0  1.0      3.0     130.0  250.0  0.0    187.0    0.0              0
4  41.0  0.0      2.0     130.0  204.0  0.0    172.0    0.0              0


## Data Analysis